# 03 — Seleção e intervenções causais

Aqui passamos de correlação para intervenção. Para cada direção candidata, removemos sua projeção do fluxo residual e perguntamos se a recusa cai; depois somamos a direção em pedidos inofensivos e perguntamos se a recusa aumenta.

Uma candidata só é aceita se também preservar razoavelmente a distribuição de saída nos exemplos inofensivos. Por fim, comparamos o efeito com uma direção aleatória de mesma norma e ortogonal à direção escolhida.

In [ ]:
# A seleção usa as direções salvas no notebook anterior e a mesma validação fixada pela semente.
from pathlib import Path
import json

from refusal_direction_replication.pipeline import run_intervention_stage

cwd = Path.cwd()
roots = (cwd, cwd.parent, cwd / "refusal_direction", Path("/content/refusal_direction"))
ROOT = next((path for path in roots if (path / "configs" / "gemma_2b.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Não foi possível localizar a raiz de refusal_direction.")
config = json.loads((ROOT / "configs" / "gemma_2b.json").read_text(encoding="utf-8"))
config

## As três perguntas causais

Para cada candidata, medimos: quanto o score de recusa cai nos pedidos nocivos após a ablação; quanto sobe nos inofensivos após a adição; e quanto a ablação altera a distribuição original nos inofensivos, via divergência KL.

O score é calculado por exemplo antes da média. Direções com KL alta, adição sem sinal de recusa ou origem nas últimas 20% das camadas são descartadas. Entre as restantes, escolhemos a que minimiza o score nocivo sob ablação.

In [ ]:
# Pontuamos as candidatas, escolhemos uma direção e executamos as intervenções e o controle aleatório.
intervention = run_intervention_stage(ROOT)
intervention_results = intervention["results"]
intervention_results["selected"]

In [ ]:
# Ordenamos as candidatas elegíveis pelo score sob ablação e mantemos os resultados por exemplo.
eligible = [
    row for row in intervention_results["candidate_scores"]
    if row["kl_divergence"] <= 0.1
    and row["addition_refusal_score"] >= 0.0
    and row["layer"] < int(row["n_layers"] * 0.8)
]
best_rows = sorted(eligible, key=lambda row: row["ablation_refusal_score"])[:8]
condition_checks = {
    name: {
        "refusal_rate": values["refusal_rate"],
        "first_flags": values["refusal_flags"][:8],
        "first_log_odds": values["per_example_refusal_log_odds"][:8],
    }
    for name, values in intervention_results["conditions"].items()
}
best_rows, condition_checks

## Leitura do resultado

Uma boa candidata reúne três sinais: remoção reduz recusa nociva, adição aumenta recusa inofensiva e a KL permanece pequena. Nenhum desses sinais isolado basta. O controle aleatório testa ainda se uma perturbação de escala equivalente produziria o mesmo efeito.

O método local de classificação por marcadores de recusa é transparente, mas mais simples que avaliadores externos usados em partes do trabalho original. Por isso preservamos também os log-odds do primeiro token e declaramos essa diferença na comparação final.